In [1]:
# MCP SCENARIO: “Smart Banking Support Assistant”
# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load,
# provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support),
# so you can see how MCP adapts to critical service environments?


!pip install groq

# IMPORTS
import json
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

tickets_db = []

# TOOL
def create_ticket(issue, priority, category):
    ticket_id = f"BK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket

# MODEL (AI ANALYSIS)
def analyze_input(user_input):

    prompt = f"""
    Classify the banking query into:
    Category: Card Services / Online Banking / Loans / Transactions
    Priority: High / Medium

    Return JSON:
    {{
        "category": "...",
        "priority": "..."
    }}

    Query: {user_input}
    """

    try:
        res = client.chat.completions.create(
            model="llama-3.1-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        data = json.loads(res.choices[0].message.content)
        return data["category"].lower(), data["priority"].lower()

    except:
        return "general", "medium"

# DECISION ENGINE
def should_call_tool(user_input):

    prompt = f"""
    Does this banking issue require escalation?
    Answer only YES or NO.

    Query: {user_input}
    """

    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return "yes" in res.choices[0].message.content.lower()

# MCP AGENT
def mcp_agent(user_input):

    print("\n🧠 Input:", user_input)

    if should_call_tool(user_input):

        print("➡️ Escalation required")

        category, priority = analyze_input(user_input)

        result = create_ticket(user_input, priority, category)

        return f"""
        ✅ Support Ticket Created!

        🎫 Ticket ID: {result['ticket_id']}
        📂 Category: {result['category']}
        ⚡ Priority: {result['priority']}
        """

    else:
        print("➡️ AI response")

        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": user_input}]
        )

        return "🤖 " + res.choices[0].message.content

# RUN LOOP
print("🏦 Smart Banking MCP Assistant (Groq)\nType 'exit' to stop\n")

while True:
    query = input("Enter your issue: ")

    if query.lower() == "exit":
        break

    print(mcp_agent(query))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.5 MB/s eta 0:00:00
🏦 Smart Banking MCP Assistant (Groq)
Type 'exit' to stop

Enter your issue: I have an issue with my credit card

🧠 Input: I have an issue with my credit card
➡️ AI response
🤖 I'm here to help you with your credit card issue. Can you please provide more details about the problem you're experiencing? For example:

1. What type of issue are you having (e.g., unauthorized transaction, disputed charge, declined payment, etc.)?
2. What credit card do you use (e.g., type of card, issuer, card number, etc.)?
3. When did the issue occur (e.g., date, time)?
4. Have you already contacted your credit card issuer or taken any steps to resolve the issue?

Sharing this information will help me better understand your situation and provide guidance or assistance in resolving the problem.
Enter your issue: My card was declined during payment

🧠 Input: My card was declined during payment
➡️ AI response
🤖 I'd be happy to help 